# VolumeFinder: prescribing divisor volumes

> **What's in this notebook?**
> This notebook introduces `VolumeFinder`, a specialised root-finder that inverts the map from Kahler moduli to divisor volumes in a toric Calabi-Yau compactification. Starting from a user-supplied target volume vector, it returns Kahler heights `h` such that the triple-intersection formula reproduces the prescribed volumes.
>
> **What this notebook covers:**
> 1. The divisor-volume problem: geometry and conventions.
> 2. Loading a toric geometry with `cytools` and extracting the triple-intersection tensor.
> 3. Constructing and running `VolumeFinder` for a first solve.
> 4. Diagnostics: convergence history and verbosity levels.
> 5. Controlling the step-taking schedule: `JumpStep` vs `FlopStep` vs the hybrid default.
> 6. Choosing a step proposal and step size: Newton, LMA, gradient, and their learning rates.
> 7. Batch mode via `swarm()`: launching many solvers in parallel.

## Outline

1. Setup
2. The divisor-volume problem
3. Loading a toric geometry
4. A first solve
5. Diagnostics
6. Step-taking schedule
7. Step proposal and step size
8. Batch mode: swarm()
9. Take-aways
10. Further reading

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cytools import Polytope

from fanroots.demo.volume_finder import VolumeFinder
from fanroots.step_taking import FlopStep, JumpStep

## 1. The divisor-volume problem

In a toric Calabi-Yau compactification the divisor volumes are determined by the triple-intersection tensor $\kappa_{ijk}$ and the Kahler moduli $t^i$ via

$$
\tau_i = \frac{1}{2}\,\kappa_{ijk}\,t^j\,t^k.
$$

The Kahler moduli $t$ are themselves linear in the *heights* $h$ of the fan triangulation:

$$
t = \mathrm{GLSM} \cdot h,
$$

where $\mathrm{GLSM}$ is the GLSM charge matrix of the geometry.

**The problem:** given a target vector $\tau^*$, find heights $h^*$ such that $\tau(h^*) = \tau^*$.

`VolumeFinder` frames this as a nonlinear root-finding problem

$$
F(h) = \mathrm{div\_vols}(h,\,\mathrm{extrapolate=True}) - \tau^* = 0,
$$

with Jacobian

$$
J(h) = (\kappa \cdot t) \cdot \mathrm{GLSM}.
$$

A subtlety worth keeping in mind: the map $h \mapsto \tau(h)$ is only smooth *inside* a given triangulation chamber of the secondary fan. When the solver moves across a chamber wall it performs a *flip* — a local change of triangulation — to stay in a valid Kahler cone. This chamber structure motivates the two-step *step-taking schedule* described in [Section 6].

## 2. Loading a toric geometry

We use a simple four-dimensional reflexive polytope as a running example. After triangulating and extracting the Calabi-Yau hypersurface, `cytools` gives us everything we need: the triple-intersection tensor `kappa`, a `VectorConfiguration` object `vc`, and a default height vector.

In [ ]:
# Define a reflexive polytope (here: the mirror of the quintic)
p = Polytope([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [-1, -1, -1, -1]])

# Triangulate and extract the Calabi-Yau
t = p.triangulate()
cy = t.get_cy()

# Triple-intersection tensor (shape: n x n x n)
kappa = cy.intersection_numbers(in_basis=True, pushed_down=True, as_np_array=True)

# VectorConfiguration from the point configuration of the triangulation
vc = t.point_configuration()

print("Polytope dimension:", p.dim())
print("Number of Kahler moduli:", cy.h11())

In [ ]:
# Default heights and the corresponding divisor volumes
h0 = vc.subdivide().heights()
print("Default heights h0 =", h0)

# Evaluate divisor volumes at the default heights
# We will use a VolumeFinder to evaluate this conveniently
vf_probe = VolumeFinder(vc, target=np.ones(cy.h11()))  # target is a placeholder here
tau0 = vf_probe.div_vols(h0, extrapolate=True)
print("Divisor volumes at h0:", tau0)

# Choose a realistic target: scale the default volumes by a factor
tau_target = 2.0 * tau0
print("Target volumes tau_target:", tau_target)

## 3. A first solve

Constructing a `VolumeFinder` requires at minimum a `VectorConfiguration` `vc` and a `target` volume vector. Additional keyword arguments are forwarded to the parent `FanRoots` class.

```
VolumeFinder(vc, target, step_taking_schedule=None, learning_rate=0.9, **kwargs)
```

Calling `optimize()` runs the solver until convergence (or until the budget is exhausted). Internally it is equivalent to `step(num=inf)`.

In [ ]:
vf = VolumeFinder(vc, target=tau_target, verbosity=1)
vf.optimize()

In [ ]:
# Inspect the solver status
status = vf.get_status()
print("finished        :", status["finished"])
print("finished_reason :", status["finished_reason"])
print("res_norm        :", status["res_norm"])       # squared norm of the residual
print("last_step_size  :", status["last_step_size"])
print("num_steps       :", status["num_steps"])
print("num_flips       :", status["num_flips"])
print("num_fct_calls   :", status["num_fct_calls"])

In [ ]:
# Compare achieved volumes with the target
h_star = vf.heights
tau_achieved = vf.div_vols(h_star, extrapolate=True)

print("Target   :", tau_target)
print("Achieved :", tau_achieved)
print("Max abs error:", np.max(np.abs(tau_achieved - tau_target)))

## 4. Diagnostics

`VolumeFinder` (via `FanRoots`) records the squared residual norm after every step in `history_res_norm`. This list is always populated regardless of `history_level`.

Setting `history_level >= 1` additionally records the full height vector `x()` after every step in `history`. Higher verbosity levels print progress to stdout:

- `verbosity=0` — silent.
- `verbosity=1` — one line per step.
- `verbosity=2` — detailed step information.

Note: `res_norm()` and the entries in `history_res_norm` store the *squared* norm, i.e. $\|F(h)\|^2$. The tolerance passed to the solver is also stored as its square internally.

In [ ]:
# Re-run with history_level=1 to also record the trajectory
vf_hist = VolumeFinder(vc, target=tau_target, history_level=1, verbosity=0)
vf_hist.optimize()

# Plot convergence on a semilogy scale
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(vf_hist.history_res_norm, color="steelblue", linewidth=2)
ax.set_xlabel("Step")
ax.set_ylabel(r"$\|F(h)\|^2$ (squared residual norm)")
ax.set_title("VolumeFinder convergence history")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Converged in {len(vf_hist.history_res_norm)} steps.")
print(f"Height trajectory has {len(vf_hist.history)} recorded vectors.")

## 5. Step-taking schedule

Inside a triangulation chamber the solver takes ordinary Newton-type steps. When the candidate point exits the current chamber it must *flip* into an adjacent one. `fanroots` models this with two step-taking primitives:

| Class | Behaviour |
|---|---|
| `JumpStep` | Accepts the full proposal immediately; fast but may overshoot. |
| `FlopStep(max_num_flips=10)` | Performs a sequence of fan flips to move smoothly across chamber walls; safer near boundaries. |

The **default `step_taking_schedule`** for `VolumeFinder` is a hybrid: it uses `JumpStep` when the last step size is large ($\geq 1$) and falls back to `FlopStep(max_num_flips=10)` for smaller, more cautious steps. This gives rapid large-scale movement early on and careful chamber-tracking near convergence.

You can override the schedule by passing a custom `step_taking_schedule` to the constructor.

In [ ]:
# Compare three strategies: JumpStep-only, FlopStep-only, and the hybrid default

schedules = {
    "JumpStep only":  [[lambda opt: True, JumpStep()]],
    "FlopStep only":  [[lambda opt: True, FlopStep(max_num_flips=10)]],
    "default (hybrid)": None,  # VolumeFinder's built-in default
}

for label, schedule in schedules.items():
    vf_s = VolumeFinder(vc, target=tau_target,
                        step_taking_schedule=schedule,
                        verbosity=0)
    vf_s.optimize()
    s = vf_s.get_status()
    print(f"{label:25s}  finished={s['finished']}  "
          f"steps={s['num_steps']:4d}  flips={s['num_flips']:4d}  "
          f"res_norm={s['res_norm']:.2e}")

## 6. Step proposal and step size

Each solver step is split into two sub-decisions:

**Step proposal** — how to compute the search direction $\delta h$:

| Key | Alias | Description |
|---|---|---|
| `"gauss_newton"` | `"newton"` | Gauss-Newton direction; default learning rate 0.1. |
| `"lma"` | — | Levenberg-Marquardt; default learning rate 0.1. |
| `"grad"` | — | Gradient descent; default learning rate 1e-3. |

**Step size** — how far to move along $\delta h$:

| Key | Description |
|---|---|
| `"shrink"` | Shrink until the step is accepted; **default for `VolumeFinder`**. |
| `"bls"` | Backtracking line search (Armijo condition). |
| `"ternary"` | Ternary search along the direction. |
| `"naive"` | Fixed step size; no line search. |

The `learning_rate` scales the initial step. `VolumeFinder` defaults to `learning_rate=0.9`, which is intentionally larger than the Newton/LMA default of 0.1 because the volume map is smooth and well-conditioned in the interior of the Kahler cone.

In [ ]:
# Compare step proposals
proposals = ["gauss_newton", "lma", "grad"]

for proposal in proposals:
    lr = 1e-3 if proposal == "grad" else 0.9
    vf_p = VolumeFinder(vc, target=tau_target,
                        step_proposal=proposal,
                        learning_rate=lr,
                        verbosity=0)
    vf_p.optimize()
    s = vf_p.get_status()
    print(f"proposal={proposal:15s}  finished={s['finished']}  "
          f"steps={s['num_steps']:4d}  res_norm={s['res_norm']:.2e}")

## 7. Batch mode: swarm()

The volume-inversion problem can be multi-modal or, in geometries with many Kahler moduli, sensitive to the starting point. `FanRoots.swarm()` creates a `BatchOptimizer` that launches `N` independent solvers from randomly perturbed starting heights:

```python
batch = vf.swarm(N, scale, verbosity=0)
```

- `N` — number of independent solvers.
- `scale` — standard deviation of the Gaussian perturbation applied to the default starting heights.
- `verbosity=0` — suppress per-solver output.

`BatchOptimizer.optimize(serial=False)` runs all solvers in parallel (when `serial=False`) and stores each individual `VolumeFinder` in `BatchOptimizer.batch`. You can then filter for converged solutions and, for example, select the one with the smallest residual.

In [ ]:
# Launch a swarm of 8 solvers with perturbation scale 0.3
vf_base = VolumeFinder(vc, target=tau_target, verbosity=0)
batch = vf_base.swarm(N=8, scale=0.3, verbosity=0)
batch.optimize(serial=False)

In [ ]:
# Filter converged solvers and summarise
converged = [opt for opt in batch.batch if opt.get_status()["finished"]]
print(f"{len(converged)} / {len(batch.batch)} solvers converged.")

if converged:
    best = min(converged, key=lambda opt: opt.get_status()["res_norm"])
    s_best = best.get_status()
    print(f"Best residual norm : {s_best['res_norm']:.2e}")
    print(f"Steps taken        : {s_best['num_steps']}")
    print(f"Flips performed    : {s_best['num_flips']}")

    tau_batch = best.div_vols(best.x(), extrapolate=True)
    print("Max abs error      :", np.max(np.abs(tau_batch - tau_target)))

## 8. Take-aways

- `VolumeFinder` inverts the triple-intersection volume map $\tau_i = \frac{1}{2}\kappa_{ijk}t^j t^k$ with $t = \mathrm{GLSM}\cdot h$, returning Kahler heights $h^*$ that reproduce a user-specified divisor-volume vector.
- The default `step_taking_schedule` is a **hybrid**: `JumpStep` for large moves and `FlopStep` near convergence, striking a balance between speed and robustness across triangulation chamber walls.
- `history_res_norm` is always recorded and provides an easy way to diagnose convergence; enabling `history_level=1` additionally stores the full solution trajectory.
- The `step_proposal` (`gauss_newton`, `lma`, `grad`) and `step_size` (`shrink`, `bls`, `ternary`, `naive`) options can be combined freely to tune the solver for a given geometry.
- `swarm(N, scale)` makes it straightforward to hedge against local optima or poor starting points by running many independent solvers in parallel and keeping the best converged result.

## 9. Further reading

- **API reference — core classes:** `fanroots` — `FanRoots`, `BatchOptimizer`, `fanroots_from_state`.
- **API reference — step proposals:** `fanroots.step_proposal` — `propose_gauss_newton`, `propose_lma`, `propose_gradient_descent`.
- **API reference — step taking:** `fanroots.step_taking` — `JumpStep`, `FlopStep`, custom schedules.
- **API reference — demo:** `fanroots.demo` — `VolumeFinder`.
- **Conceptual background:** see the *Introduction* chapters — the secondary fan, Kähler moduli, and the algorithm.
- **Original paper:** N. MacFadden and L. McAllister, *Root-finding on secondary fans*, [arXiv:2406.13751](https://arxiv.org/abs/2406.13751).